In [338]:
import os
from rdflib import Graph
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from kneed import KneeLocator

from elasticsearch import Elasticsearch

from utils import ollama_request, EMBEDD_MODEL_1

import spacy
nlp = spacy.load("en_core_web_sm")

INDEX_NAME = os.getenv("INDEX_NAME")
ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
FRAGMENT_INDEX_NAME = f"{INDEX_NAME}_fragments"
TRIPLETS_INDEX_NAME = f"{INDEX_NAME}_triplets_index"

EMBEDDING_MODEL = EMBEDD_MODEL_1

es_client = Elasticsearch('http://localhost:9200')

g = Graph()
g.parse(f"{INDEX_NAME}.ttl", format="turtle")

<Graph identifier=Nf7ad1c2abf204a7f9ddedda2e3cba5db (<class 'rdflib.graph.Graph'>)>

In [339]:
def extract_objects(text):
    doc = nlp(text)

    subjects = []
    target_deps = {
        "nsubj",
        "nsubjpass",
        "dobj",
        "pobj",
        "iobj",
    }
    for token in doc:
        if token.dep_ in target_deps:
            subject = " ".join(
                t.text for t in token.subtree
            )
            subjects.append(subject)

    return subjects

In [340]:
def execute_query(graph, start_date, end_date, filtering_criteria):
    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id ?triplet_id
WHERE {{
    ?stmt a rdf:Statement ;
          rdf:subject ?s ;
          rdf:predicate ?p ;
          rdf:object ?o ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:triplet_id ?triplet_id ;
          ns1:speech_id ?speech_id .

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
        {filtering_criteria}
    )
}}
""".format(
    start_date=start_date,
    end_date=end_date,
    filtering_criteria=filtering_criteria
)
    return graph.query(query)

In [341]:
def get_df_from_query_result(query_res):
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
    }
    for row in query_res:
        fragment_start = row.start
        fragment_end = row.end
        speech_id = row.speech_id

        triplets_resp["subject"].append(row.s.split('/')[-1])
        triplets_resp["predicate"].append(row.p.split('/')[-1])
        triplets_resp["object"].append(row.o.split('/')[-1])
        triplets_resp["date"].append(str(row.date.value))
        triplets_resp["start"].append(fragment_start)
        triplets_resp["end"].append(fragment_end)
        triplets_resp["speech_id"].append(speech_id)
    return pd.DataFrame(triplets_resp)

In [342]:
def get_triplets_by_id(graph, triplet_ids, start_date="1900-01-01", end_date="2100-01-01"):
    triplet_ids_values = " ".join(str(x) for x in triplet_ids)

    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id
WHERE {{
    ?stmt a rdf:Statement ;
          rdf:subject ?s ;
          rdf:predicate ?p ;
          rdf:object ?o ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:speech_id ?speech_id ;
          ns1:triplet_id ?triplet_id .

    VALUES ?triplet_id {{ {triplet_ids_values} }}
    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        triplet_ids_values=triplet_ids_values,
    )

    return graph.query(query)

In [343]:
def knee_cutoff(scores):
    scores = np.sort(np.asarray(scores))[::-1]

    indices = np.arange(len(scores))

    knee = KneeLocator(
        indices,
        scores,
        curve="convex",
        direction="decreasing"
    )

    return int(knee.knee) - 1

In [344]:
def score_df_cutoff(df, top_k=10):
    df = df.iloc[:top_k, :].sort_values(by="score", ascending=False)

    sorted_scores = df["score"].values
    if len(sorted_scores) < 2:
        return df
    gaps = sorted_scores[:-1] - sorted_scores[1:]
    cutoff_1 = np.argmax(gaps) + 1
    cutoff_2 = knee_cutoff(sorted_scores)

    if cutoff_2 == 0:
        cutoff = cutoff_1
    else:
        cutoff = cutoff_2
    return df.iloc[:cutoff, :]

In [345]:
def find_similar_entities(entity, score_threshold=0.5, top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(entity)
    final_res = {
        "score": [],
        "value_name": [],
    }
    response = es_client.search(
        index=ENT_INDEX_NAME,
        size=100,
        knn={
            "field": "value_embedding",
            "query_vector": query_embedding.tolist(),
            "k": 100,
            "num_candidates": 100,
            "filter": {
                "match_all": {}
            }
        },
        source=["value_name"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["value_name"].append(result["_source"]["value_name"])
    
    final_res = pd.DataFrame(final_res)
    return score_df_cutoff(final_res, top_k=top_k)


In [346]:
def find_question_related_triplets(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01", top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(question)
    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": query_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "match_all": {}
            }
        },
        source=["triplet_id"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    related_triplets = pd.DataFrame(final_res)
    related_triplets = score_df_cutoff(related_triplets, top_k=top_k)
    related_ids = list(related_triplets["triplet_id"])

    query_res = get_triplets_by_id(graph, related_ids, start_date=start_date, end_date=end_date)
    res_df = get_df_from_query_result(query_res)
    res_df["information_type"] = "main"

    return res_df


In [347]:
def find_additional_related_triplets(graph, question, score_threshold=0.5, top_entity_k=10, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    question_objects = extract_objects(question)
    question_embedding = EMBEDDING_MODEL.encode(question)
    similar_entities = []
    for object in question_objects:
        similar_entities = similar_entities + list(find_similar_entities(object, score_threshold=score_threshold, top_k=top_entity_k)["value_name"].values)

    filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
        "triplet_id": [],
    } 
    for ent in similar_entities:
        try:
            query_res = execute_query(
                graph=graph,
                start_date=start_date,
                end_date=end_date,
                filtering_criteria=filtering_criteria.format(val=ent),
            )
        except Exception as e:
            continue

        for row in query_res:
            fragment_start = row.start
            fragment_end = row.end
            speech_id = row.speech_id

            triplet_id = row.triplet_id
            
            triplets_resp["subject"].append(row.s.split('/')[-1])
            triplets_resp["predicate"].append(row.p.split('/')[-1])
            triplets_resp["object"].append(row.o.split('/')[-1])
            triplets_resp["date"].append(str(row.date.value))
            triplets_resp["start"].append(fragment_start)
            triplets_resp["end"].append(fragment_end)
            triplets_resp["speech_id"].append(speech_id)
            triplets_resp["triplet_id"].append(triplet_id)

    triplets_df = pd.DataFrame(triplets_resp)

    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": question_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "terms": {
                    "triplet_id": triplets_df["triplet_id"].tolist()
                }
            }
        },
        source=["triplet_id"]
    )
    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    final_scores = pd.DataFrame(final_res)
    final_scores["triplet_id"] = final_scores["triplet_id"].astype("int64")
    triplets_df["triplet_id"] = triplets_df["triplet_id"].astype("int64")

    triplets_df = triplets_df.merge(
        final_scores,
        on="triplet_id",
        how="inner"
    )
    triplets_df = score_df_cutoff(triplets_df, top_k=top_triplet_k)

    triplets_df["information_type"] = "additional"
    triplets_df = triplets_df.drop(columns=["score", "triplet_id"])
    return triplets_df

In [348]:
def get_prompt(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01"):

    main_triplets_related_to_question = find_question_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_k=1000
    )

    additional_triplets_related_to_question = find_additional_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_entity_k=10,
        top_triplet_k=1000
    )

    final_triplets = pd.concat([main_triplets_related_to_question, additional_triplets_related_to_question], ignore_index=True)

    final_triplets = final_triplets.drop_duplicates(subset=["subject", "predicate", "object", "date", "start", "end", "speech_id"])
    final_triplets = final_triplets.sort_values(by=["date", "start"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    main_triplets = final_triplets[final_triplets["information_type"] == "main"]
    additional_triplets = final_triplets[final_triplets["information_type"] == "additional"]

    prompt = QUESTION_ANSWER_PROMPT.format(
        question=question,
        main_facts=main_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records"),
        additional_facts=additional_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")
    )

    return prompt, final_triplets

In [355]:
def get_fragment(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    start = int(row.start)
    end = int(row.end) + 1
    speech_id = row.speech_id
    return ".".join(es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"].split(".")[start:end])

def get_speech(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    speech_id = row.speech_id

    return es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

In [350]:
QUESTION_ANSWER_PROMPT = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.
Each fact contains an index.
You can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".
Use just syntax with [index] to refer to the fact, do not use any additional words.

Facts are split into two categories: main and additional.
Main facts (most important) are directly related to the research question. 
Additional facts provide context or background information for entities present in the question.

Research question: {question}

Main Facts:
{main_facts}

Additional Facts:
{additional_facts}

Your task is to answer the research question based on the provided facts."
"""

In [357]:
QUESTION = """
What are Putin’s most common arguments for strengthening the army?
"""

START_DATE = "1999-01-01"
END_DATE = "2024-12-31"

PROMPT, final_triplets= get_prompt(
    graph=g,
    question=QUESTION,
    score_threshold=0.5,
    start_date=START_DATE,
    end_date=END_DATE
)
print(f"Used main facts: {len(final_triplets[final_triplets['information_type'] == 'main'])}")
print(f"Used additional facts: {len(final_triplets[final_triplets['information_type'] == 'additional'])}")
PROMPT

Used main facts: 34
Used additional facts: 4


'\nYou are a political scientist model. You are given a research question and a set of facts from a knowledge graph.\nEach of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.\nEach of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.\nThe triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.\nEach fact contains an index.\nYou can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".\nUse just syntax with [index] to refer to the fact, do not use any additional words.\n\nFacts are split into two categories: main and additional.\nMain facts (most important) are directly related to the researc

In [ ]:
res = ollama_request(
    prompt=PROMPT, is_stream=False
)
 
display(Markdown(res))

In [362]:
REFERENCE_NUM = 10

display(final_triplets[final_triplets["index"] == REFERENCE_NUM][["subject", "predicate", "object"]].reset_index(drop=True))

display(Markdown("### Reference Fragment"))
display(Markdown(get_fragment(final_triplets, REFERENCE_NUM)))

display(Markdown("### Reference Speech"))
display(Markdown(get_speech(final_triplets, REFERENCE_NUM)))


,subject,predicate,object
0,vladimir_putin,focuses_on_war_lessons,balance_between_strength_and_reason


### Reference Fragment

 Their victory and the heritage it represents still serve us today, helping us to overcome difficulties and move forward, calling us to new action and glorious new achievements. Happy holiday, dear friends, happy Victory Day!. Glory to the soldiers of 1945! Glory to the victorious people! Glory to Russia

### Reference Speech

I greet the veterans of the Great Patriotic War!  You have defended our right to be citizens of a free country. I thank you for your labours in peace and war, for your courage and endurance and for being here beside us today.  Every year we honour your victorious generation with emotion in our hearts. It is our sacred duty to cherish and care for those who fought and those who toiled on the home front during those terrible times of war.  * * * We know only too well the price of victory and we remember the millions who lost their lives, the millions who were maimed and also the millions who were not born after that terrible war. We remember the devastated cities, the burned villages and the destruction of our national cultural treasures. We remember everything we lost during those days.  May 9 is truly a time of celebration with “tears in our eyes”. It is a celebration in which grandeur and sorrow, national pride and national memory, the bright glint of medals and the tears of veterans are forever bound together.  * * * Dear comrades! We did not give in back then in 1945. Our people showed such unity and strength of will that we roused the entire world to join the struggle against fascism. We do not have the right to lose this spirit and to betray our victories that we all consider sacred. Today, as during the war years, the red banner of our Armed Forces flies proud once again.  Traditions of victory form the spiritual backbone of the Russian Army – an army that is developing, modernising and adapting to new demands.  * * * We achieved victory in the most just war of the twentieth century, a war of liberation for the sovereignty and independence of our Motherland. We need to remember the lessons of this war. They teach us to look for a balance between strength and reason and they warn us that becoming an accomplice of violence and extremism has immensely tragic consequences. Our entire post-war history teaches us that no country can build a safer world for itself alone, and even more so, cannot build its security to the detriment of others.  Dear citizens of Russia! Victory Day has been our most important celebration, the celebration most cherished by our people and held most dear to our hearts for 56 years now. Not every people can count such a victory among its achievements. To be the inheritors of this victory is not just a great honour but is also a great responsibility.  The heritage of our frontline soldiers flows in our blood today and their feat lives on in our hearts. Their victory and the heritage it represents still serve us today, helping us to overcome difficulties and move forward, calling us to new action and glorious new achievements. Happy holiday, dear friends, happy Victory Day!. Glory to the soldiers of 1945! Glory to the victorious people! Glory to Russia.  Hurrah! 